In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import FunctionTransformer
from bs4 import BeautifulSoup

In [3]:
from pathlib import Path

TRAIN_PATH = Path("../data/processed/train.csv")

train_df = pd.read_csv(TRAIN_PATH)

X_train = train_df["review"]
y_train = train_df["sentiment"]

print("Training reviews:", len(X_train))
print("Training labels:", y_train.value_counts())

Training reviews: 40000
Training labels: sentiment
positive    20000
negative    20000
Name: count, dtype: int64


In [4]:
def strip_html_batch(texts):
    return [
        BeautifulSoup(text, "html.parser").get_text(separator=" ")
        for text in texts
    ]


baseline = Pipeline([
    (
        "html",
        FunctionTransformer(
            strip_html_batch,
            validate=False
        )
    ),
    (
        "vectorizer",
        CountVectorizer()
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

print("Baseline pipeline created successfully!")

Baseline pipeline created successfully!


In [5]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    baseline,
    X_train,
    y_train,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

print("Fold accuracies:", np.round(scores, 4))
print(f"Mean CV accuracy: {scores.mean():.4f}")
print(f"Standard deviation: {scores.std():.4f}")

Fold accuracies: [0.883  0.8838 0.8886 0.8914 0.88  ]
Mean CV accuracy: 0.8854
Standard deviation: 0.0041


In [6]:
import re
import string

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from bs4 import BeautifulSoup


def strip_html_batch(texts):
    return [
        BeautifulSoup(text, "html.parser").get_text(separator=" ")
        for text in texts
    ]


def lowercase_batch(texts):
    return [text.lower() for text in texts]


def remove_punctuation_batch(texts):
    return [
        re.sub(r"[^\w\s]", " ", text)
        for text in texts
    ]


# Keep negation words to preserve sentiment
custom_stop_words = list(
    ENGLISH_STOP_WORDS - {"not", "no", "never"}
)


def remove_stopwords_batch(texts):
    cleaned = []

    for text in texts:
        words = text.split()
        words = [
            word for word in words
            if word.lower() not in custom_stop_words
        ]
        cleaned.append(" ".join(words))

    return cleaned


print("Preprocessing functions ready!")

Preprocessing functions ready!


In [7]:
from snowballstemmer import stemmer

english_stemmer = stemmer("english")

def stem_text_batch(texts):
    return [
        " ".join(english_stemmer.stemWords(str(text).lower().split()))
        for text in texts
    ]

print(stem_text_batch([
    "The movies were amazing and entertaining",
    "The performances were disappointing"
]))

['the movi were amaz and entertain', 'the perform were disappoint']


Raw text: 0.8869 ± 0.0013
HTML stripping: 0.8873 ± 0.0006
Lowercasing: 0.8873 ± 0.0006
Punctuation removal: 0.8873 ± 0.0006
Stop-word removal: 0.8800 ± 0.0027


,Variant,Mean CV Accuracy,Std Dev
0,Raw text,0.88685,0.001263
1,HTML stripping,0.88730,0.000640
2,Lowercasing,0.88730,0.000640
3,Punctuation removal,0.88730,0.000640
4,Stop-word removal,0.87995,0.002745
